# Vocabulary Tools
> Vocabulary tools for working with semantic data sources.

This module provides tools for LLM agents to work with semantic vocabularies and JSON-LD data.
It augments the retriever and memory modules with agent-centric functions for working with
vocabularies, handling JSON-LD contexts, and navigating semantic data sources.

Key features:
- Support for vocabularies that don't follow standard linked data principles
- Multi-level support for different types of vocabularies
- Helper functions for creating and working with datasets
- Tools for compacting and expanding JSON-LD data

In [ ]:
#| default_exp vocabtools

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import json
import httpx
from pyld import jsonld
from typing import Dict, List, Optional, Any, Union
from rdflib import Graph

In [ ]:
#| export
# Registry of vocabulary information
VOCABULARY_REGISTRY = {
    "schema": {
        "uri": "https://schema.org/",
        "support_level": "direct",
        "ttl_location": "https://schema.org/version/latest/schemaorg-current-https.ttl",
        "context_location": "https://schema.org/docs/jsonldcontext.jsonld",
        "access_pattern": "link_header_to_context",
        "inline_context": False
    },
    "croissant": {
        "uri": "http://mlcommons.org/croissant/",
        "support_level": "direct",
        "ttl_location": "https://raw.githubusercontent.com/mlcommons/croissant/main/docs/croissant.ttl",
        "context_location": "https://raw.githubusercontent.com/mlcommons/croissant/main/docs/context.jsonld",
        "access_pattern": "github_raw",
        "inline_context": True
    },
    "ro-crate": {
        "uri": "https://w3id.org/ro/crate/1.2-DRAFT/context",
        "support_level": "cache",
        "ttl_location": None,
        "context_location": "https://w3id.org/ro/crate/1.2-DRAFT/context",
        "access_pattern": "persistent_uri",
        "inline_context": False
    }
}

## Core functions for vocabulary-aware document loading

In [ ]:
#| export
def create_vocab_aware_document_loader(registry=None):
    """Create a document loader that handles vocabularies at different support levels.
    
    Args:
        registry: Optional custom registry to use
        
    Returns:
        A document loader function for PyLD
    """
    from pyld import jsonld
    
    # Use default registry if none provided
    registry = registry or VOCABULARY_REGISTRY
    
    # Get the default document loader to fall back to
    default_loader = jsonld.get_document_loader()
    
    # Create an in-memory cache for contexts
    context_cache = {}
    
    def vocab_aware_loader(url, options=None):
        """Custom document loader that handles different vocabulary support levels"""
        options = options or {}
        
        # Check if this URL matches any known vocabulary
        for vocab_name, vocab_info in registry.items():
            vocab_uri = vocab_info["uri"]
            
            if url == vocab_uri or url.startswith(vocab_uri):
                support_level = vocab_info.get("support_level", "discover")
                
                # Handle based on support level
                if support_level == "direct":
                    return _handle_direct_support(url, vocab_info, context_cache)
                elif support_level == "cache":
                    return _handle_cache_support(url, vocab_info, context_cache, default_loader)
                else:  # "discover"
                    return _handle_discovery_support(url, default_loader, context_cache)
        
        # No match found, use default loader
        return default_loader(url, options)
    
    return vocab_aware_loader

In [ ]:
#| export
def _handle_direct_support(url, vocab_info, context_cache):
    """Handle vocabularies that need direct intervention.
    
    Args:
        url: The URL being requested
        vocab_info: Information about the vocabulary from the registry
        context_cache: Cache for storing retrieved contexts
        
    Returns:
        A document object for PyLD
    """
    import httpx
    import json
    
    # Check if we've already cached this context
    if url in context_cache:
        return context_cache[url]
    
    # Get the context location from vocab info
    context_location = vocab_info.get("context_location")
    
    if context_location:
        try:
            # Fetch the context from the correct location
            response = httpx.get(context_location)
            if response.status_code == 200:
                try:
                    context_data = response.json()
                    
                    # Create the document object that PyLD expects
                    result = {
                        'contextUrl': None,
                        'documentUrl': url,
                        'document': context_data
                    }
                    
                    # Cache the result
                    context_cache[url] = result
                    return result
                except Exception as e:
                    print(f"Error parsing context JSON from {context_location}: {e}")
        except Exception as e:
            print(f"Error fetching context from {context_location}: {e}")
    
    # If we have an inline context example, use that as fallback
    if vocab_info.get("inline_context"):
        # In a real implementation, we would extract this from examples
        # For now, we'll use a simplified version
        default_context = {
            "@vocab": vocab_info["uri"],
            # Add other key terms for this vocabulary
        }
        
        result = {
            'contextUrl': None,
            'documentUrl': url,
            'document': default_context
        }
        
        # Cache the result
        context_cache[url] = result
        return result
    
    # If all else fails, return a minimal context
    minimal_context = {
        "@vocab": vocab_info["uri"]
    }
    
    result = {
        'contextUrl': None,
        'documentUrl': url,
        'document': minimal_context
    }
    
    # Cache the result
    context_cache[url] = result
    return result

In [ ]:
#| export
def _handle_cache_support(url, vocab_info, context_cache, default_loader):
    """Handle vocabularies that can be dereferenced but benefit from caching.
    
    Args:
        url: The URL being requested
        vocab_info: Information about the vocabulary from the registry
        context_cache: Cache for storing retrieved contexts
        default_loader: The default document loader to use
        
    Returns:
        A document object for PyLD
    """
    # Check if we've already cached this context
    if url in context_cache:
        return context_cache[url]
    
    # Try to use the default loader to fetch the context
    try:
        result = default_loader(url, {})
        
        # Cache the successful result
        context_cache[url] = result
        return result
    except Exception as e:
        print(f"Error using default loader for {url}: {e}")
        
        # If default loader fails but we have a known context location, try that
        context_location = vocab_info.get("context_location")
        if context_location and context_location != url:
            try:
                # Try to load from the specific context location
                result = default_loader(context_location, {})
                
                # Update the documentUrl to the original URL
                result['documentUrl'] = url
                
                # Cache the result
                context_cache[url] = result
                return result
            except Exception as inner_e:
                print(f"Error fetching from context_location {context_location}: {inner_e}")
    
    # If all attempts fail, create a minimal context
    minimal_context = {
        "@vocab": vocab_info.get("uri", url)
    }
    
    result = {
        'contextUrl': None,
        'documentUrl': url,
        'document': minimal_context
    }
    
    # Cache the result
    context_cache[url] = result
    return result

In [ ]:
#| export
def _handle_discovery_support(url, default_loader, context_cache):
    """Handle unknown vocabularies by attempting to discover their structure.
    
    Args:
        url: The URL being requested
        default_loader: The default document loader to use
        context_cache: Cache for storing retrieved contexts
        
    Returns:
        A document object for PyLD
    """
    # Check if we've already cached this context
    if url in context_cache:
        return context_cache[url]
    
    # Try standard dereferencing first
    try:
        result = default_loader(url, {})
        
        # Cache the successful result
        context_cache[url] = result
        return result
    except Exception as e:
        print(f"Standard dereferencing failed for {url}: {e}")
    
    # Try common variations of the URL
    variations = [
        f"{url}/context",                # Common pattern: append /context
        f"{url.rstrip('/')}/context.jsonld",  # Common pattern: append /context.jsonld
        f"{url}/latest/context",         # Version pattern: /latest/context
        f"{url.rstrip('/')}/.well-known/context.jsonld"  # Well-known pattern
    ]
    
    for variation in variations:
        try:
            result = default_loader(variation, {})
            
            # Update the documentUrl to the original URL
            result['documentUrl'] = url
            
            # Cache the successful result
            context_cache[url] = result
            return result
        except Exception as e:
            print(f"Variation {variation} failed: {e}")
    
    # If all discovery attempts fail, create a minimal context
    minimal_context = {
        "@vocab": url
    }
    
    result = {
        'contextUrl': None,
        'documentUrl': url,
        'document': minimal_context
    }
    
    # Cache the result
    context_cache[url] = result
    return result

In [ ]:
#| export
def register_vocab_aware_loader():
    """Register our vocabulary-aware document loader with PyLD.
    
    This function creates and registers a custom document loader with PyLD
    that handles vocabularies according to their support level in the registry.
    
    Returns:
        The created document loader function
    """
    from pyld import jsonld
    
    # Create and register the loader
    loader = create_vocab_aware_document_loader()
    jsonld.set_document_loader(loader)
    
    return loader

## Helper functions for working with vocabularies

In [ ]:
#| export
def load_context_for_vocabulary(vocab_name):
    """Load and return the context for a specific vocabulary.
    
    This function retrieves the context for a vocabulary, handling different
    support levels appropriately.
    
    Args:
        vocab_name: Name of the vocabulary to load context for
        
    Returns:
        The JSON-LD context for the vocabulary
        
    Raises:
        ValueError: If the vocabulary is not found in the registry
    """
    # Check if the vocabulary is in our registry
    if vocab_name not in VOCABULARY_REGISTRY:
        raise ValueError(f"Unknown vocabulary: {vocab_name}")
    
    vocab_info = VOCABULARY_REGISTRY[vocab_name]
    vocab_uri = vocab_info["uri"]
    
    # Create a temporary cache
    context_cache = {}
    
    # Handle based on support level
    support_level = vocab_info.get("support_level", "discover")
    
    if support_level == "direct":
        result = _handle_direct_support(vocab_uri, vocab_info, context_cache)
    elif support_level == "cache":
        from pyld import jsonld
        default_loader = jsonld.get_document_loader()
        result = _handle_cache_support(vocab_uri, vocab_info, context_cache, default_loader)
    else:  # "discover"
        from pyld import jsonld
        default_loader = jsonld.get_document_loader()
        result = _handle_discovery_support(vocab_uri, default_loader, context_cache)
    
    # Return just the context document
    return result["document"]

In [ ]:
#| export
def create_dataset_with_vocabulary(dataset_data, vocab_name):
    """Create a dataset using a specific vocabulary.
    
    This function adds the appropriate context for a vocabulary to a dataset.
    
    Args:
        dataset_data: Dataset properties without the @context
        vocab_name: Name of the vocabulary to use
        
    Returns:
        The dataset with proper context
        
    Example:
        >>> dataset_data = {
        ...     "@type": "Dataset",
        ...     "@id": "https://example.org/datasets/my-dataset",
        ...     "name": "My Dataset",
        ...     "description": "A sample dataset"
        ... }
        >>> dataset = create_dataset_with_vocabulary(dataset_data, "schema")
    """
    # Load the context for the vocabulary
    context = load_context_for_vocabulary(vocab_name)
    
    # Create the dataset with the context
    dataset = {
        "@context": context,
        **dataset_data
    }
    
    return dataset

In [ ]:
#| export
def add_dataset_to_memory(memory, dataset_data, vocab_name):
    """Add a dataset to semantic memory with proper vocabulary handling.
    
    This function creates a dataset with the appropriate context and adds it to
    semantic memory.
    
    Args:
        memory: SemanticMemory instance
        dataset_data: Dataset properties without the @context
        vocab_name: Name of the vocabulary to use
        
    Returns:
        The expanded dataset
        
    Example:
        >>> memory = SemanticMemory()
        >>> dataset_data = {
        ...     "@type": "Dataset",
        ...     "@id": "https://example.org/datasets/my-dataset",
        ...     "name": "My Dataset"
        ... }
        >>> expanded = add_dataset_to_memory(memory, dataset_data, "schema")
    """
    # Register our vocabulary-aware document loader
    register_vocab_aware_loader()
    
    # Create the dataset with the vocabulary context
    dataset = create_dataset_with_vocabulary(dataset_data, vocab_name)
    
    # Add to memory
    return memory.add_jsonld(dataset)

In [ ]:
#| export
def compact_entity_with_vocabulary(entity, vocab_name):
    """Compact an entity using the proper context for a vocabulary.
    
    This function retrieves an entity from memory and compacts it using the
    appropriate context for a vocabulary.
    
    Args:
        entity: The entity to compact (typically from memory.query_by_id)
        vocab_name: Name of the vocabulary to use for compaction
        
    Returns:
        The compacted entity
        
    Example:
        >>> memory = SemanticMemory()
        >>> # ... add entity to memory ...
        >>> entity = memory.query_by_id("https://example.org/datasets/my-dataset")
        >>> compacted = compact_entity_with_vocabulary(entity, "schema")
        >>> print(compacted["name"])  # Access property by simple name
    """
    from pyld import jsonld
    
    # Get the proper context object for this vocabulary
    context = load_context_for_vocabulary(vocab_name)
    
    # Compact using this context
    compacted = jsonld.compact(entity, context)
    
    return compacted

In [ ]:
#| export
def example_croissant_usage(memory):
    """Example of using a CROISSANT dataset with our enhanced vocabulary tools.
    
    Args:
        memory: SemanticMemory instance
        
    Returns:
        The dataset entity ID
    """
    # Create a minimal CROISSANT dataset
    dataset_data = {
        "@type": "Dataset",
        "@id": "https://example.org/datasets/my-ml-dataset",
        "name": "Example ML Dataset",
        "description": "A dataset for machine learning experiments",
        "conformsTo": "http://mlcommons.org/croissant/1.1"
    }
    
    # Add to memory using our vocabulary tools
    expanded = add_dataset_to_memory(memory, dataset_data, "croissant")
    
    # Return the dataset ID
    return dataset_data["@id"]

In [ ]:
#| export
def test_complete_workflow():
    """Test the complete workflow from adding a dataset to querying and compacting it."""
    from cogitarelink.memory import SemanticMemory
    
    print("=== Testing Complete Workflow ===")
    
    # Initialize components
    memory = SemanticMemory()
    
    # Register our vocabulary-aware document loader
    register_vocab_aware_loader()
    
    # 1. Test with CROISSANT
    print("\n1. Testing with CROISSANT vocabulary:")
    
    # Create a CROISSANT dataset with more properties
    croissant_data = {
        "@type": "Dataset",
        "@id": "https://example.org/datasets/ml-dataset-workflow",
        "name": "ML Dataset for Workflow Test",
        "description": "Testing the complete workflow with CROISSANT",
        "conformsTo": "http://mlcommons.org/croissant/1.1",
        "keywords": ["machine learning", "test", "workflow"],
        "creator": {
            "@type": "Person",
            "name": "Test Researcher"
        }
    }
    
    # Create the dataset with vocabulary
    croissant_dataset = create_dataset_with_vocabulary(croissant_data, "croissant")
    
    # Add to memory
    memory.add_jsonld(croissant_dataset)
    print("Added CROISSANT dataset to memory")
    
    # Query from memory
    dataset_from_memory = memory.query_by_id(croissant_data["@id"])
    print(f"Retrieved dataset from memory: {croissant_data['@id']}")
    
    # Compact using our helper function
    compacted_dataset = compact_entity_with_vocabulary(dataset_from_memory, "croissant")
    
    # Verify properties
    print(f"Dataset name: {compacted_dataset.get('name', 'Unknown')}")
    print(f"Dataset type: {compacted_dataset.get('@type', 'Unknown')}")
    print(f"Dataset keywords: {compacted_dataset.get('keywords', 'Unknown')}")
    
    if "creator" in compacted_dataset:
        creator = compacted_dataset["creator"]
        print(f"Creator type: {creator.get('@type', 'Unknown')}")
        print(f"Creator name: {creator.get('name', 'Unknown')}")
    
    # 2. Test with RO-Crate
    print("\n2. Testing with RO-Crate vocabulary:")
    
    # Create an RO-Crate dataset
    rocrate_data = {
        "@type": "Dataset",
        "@id": "https://example.org/datasets/research-data-workflow",
        "name": "Research Dataset for Workflow Test",
        "description": "Testing the complete workflow with RO-Crate",
        "datePublished": "2023-11-15",
        "author": {
            "@type": "Person",
            "name": "Test Author"
        }
    }
    
    # Create the dataset with vocabulary
    rocrate_dataset = create_dataset_with_vocabulary(rocrate_data, "ro-crate")
    
    # Add to memory
    memory.add_jsonld(rocrate_dataset)
    print("Added RO-Crate dataset to memory")
    
    # Query from memory
    rocrate_from_memory = memory.query_by_id(rocrate_data["@id"])
    print(f"Retrieved dataset from memory: {rocrate_data['@id']}")
    
    # Compact using our helper function
    compacted_rocrate = compact_entity_with_vocabulary(rocrate_from_memory, "ro-crate")
    
    # Verify properties
    print(f"Dataset name: {compacted_rocrate.get('name', 'Unknown')}")
    print(f"Dataset type: {compacted_rocrate.get('@type', 'Unknown')}")
    print(f"Date published: {compacted_rocrate.get('datePublished', 'Unknown')}")
    
    if "author" in compacted_rocrate:
        author = compacted_rocrate["author"]
        print(f"Author type: {author.get('@type', 'Unknown')}")
        print(f"Author name: {author.get('name', 'Unknown')}")
    
    # 3. Test memory enhancements
    print("\n3. Testing memory enhancements:")
    
    # Test query_and_compact
    if hasattr(memory, 'query_and_compact'):
        print("\nTesting query_and_compact:")
        compacted_dataset = memory.query_and_compact(croissant_data["@id"], "croissant")
        print(f"Retrieved and compacted dataset: {compacted_dataset.get('name', 'Unknown')}")
    
    # Test detect_vocabulary
    if hasattr(memory, 'detect_vocabulary'):
        print("\nTesting detect_vocabulary:")
        croissant_vocab = memory.detect_vocabulary(croissant_data["@id"])
        print(f"Detected vocabulary for CROISSANT dataset: {croissant_vocab}")
        
        rocrate_vocab = memory.detect_vocabulary(rocrate_data["@id"])
        print(f"Detected vocabulary for RO-Crate dataset: {rocrate_vocab}")
    
    print("\n=== Workflow Test Complete ===")
    
    return memory


In [ ]:
mem = test_complete_workflow()

=== Testing Complete Workflow ===

1. Testing with CROISSANT vocabulary:
Added CROISSANT dataset to memory
Retrieved dataset from memory: https://example.org/datasets/ml-dataset-workflow
Dataset name: ML Dataset for Workflow Test
Dataset type: Dataset
Dataset keywords: ['machine learning', 'test', 'workflow']
Creator type: Person
Creator name: Test Researcher

2. Testing with RO-Crate vocabulary:
Added RO-Crate dataset to memory
Retrieved dataset from memory: https://example.org/datasets/research-data-workflow
Dataset name: Research Dataset for Workflow Test
Dataset type: Dataset
Date published: 2023-11-15
Author type: Person
Author name: Test Author

3. Testing memory enhancements:

=== Workflow Test Complete ===


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()